## Model Profiling
Profiling the models using the model.profile function by providing the compiled .tflite or .zip model path and automatically uploading profiling artifacts and history logs to Databricks Unity Catalog Volumes path specified.

In [0]:
%pip install "/Workspace/Users/raansari@silabs.com/sml_ops-0.1.0-py3-none-any.whl"
dbutils.library.restartPython()

In [0]:
from pathlib import Path
import subprocess
import shutil

mvp_profiler_path = Path("/Workspace/Users/raansari@silabs.com/mvp_profiler")

if not mvp_profiler_path.is_file():
    subprocess.run(["wget", "https://github.com/SiliconLabsSoftware/aiml-extension/raw/refs/heads/main/tool/profiler/mvp_profiler"])
    subprocess.run(["chmod", "+x", "mvp_profiler"])
    shutil.move("mvp_profiler", "/Workspace/Users/raansari@silabs.com/mvp_profiler")

### Global Credentials Configuration

Call this ONLY if you have NOT already configured the global credentials. If you already called data.config() earlier (e.g., during data ingestion)you DO NOT need to call it again before profiling as both data and model modules reuse the same config.

In [0]:
from sml.ops import data
data.config(
    server_endpoint="7405615984054316.zerobus.southcentralus.azuredatabricks.net",
    workspace_url="https://adb-7405615984054316.16.azuredatabricks.net",
    table_name="mlops_dev.default.stream_audio_events",
    client_id="2cf06ebd-b4ad-40f8-91be-74e576e147f5",
    client_secret="dose3ecfb65d12bdc5a1a8d228e140e34883"
)

In [0]:
from sml.ops import model
model_path = "/Workspace/Users/raansari@silabs.com/models/Custom_model.tflite"
print("\n--- [A] Local Simulation & Cloud Upload ---")
try:
    result = model.profile(
        model_path=model_path,
        use_simulator=True,          # Set to True to run the model profiling locally using the simulator
        volume_path="/Volumes/mlops_dev/default/profiling_results",
        profiler_path="/Workspace/Users/raansari@silabs.com/mvp_profiler"
    )
    # The result object contains all the extracted data:
    print(f"  \u2713 Model:         {result.model_name}")
    print(f"  \u2713 Arena Size:    {result.arena_size_kb:.1f} KB")
    print(f"  \u2713 Total MACs:    {result.total_macs:,}")
    print(f"  \u2713 Remote Folder: {result.output_dir}")
    print(f"  \u2713 History Log:   {result.history_log_path}")
except Exception as e:
    # If there is a failure, the script will crash here
    # but the history.log will STILL upload to the volume path.
    print(f"  [!] Profiling failed -> {e}")

### MLflow Integration
Log profiling metrics, parameters, and artifacts to MLflow for tracking and comparing model profiling runs across different models and configurations. The profiling result object from the cell above is used to populate the MLflow run.

In [0]:
import mlflow
import json
import os

# ── Configure MLflow to use Unity Catalog ──
mlflow.set_registry_uri("databricks-uc")

# Experiment lives in workspace; artifacts are stored in the volume
experiment_name = "/Workspace/Users/raansari@silabs.com/model_profiling_experiments"
volume_path = "dbfs:/Volumes/mlops_dev/default/profiling_results"

try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=volume_path
    )
except mlflow.exceptions.MlflowException as e:
    print(e)
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_id=experiment_id)

# ── Parse the profiling report for detailed metrics ──
report_path = os.path.join(result.output_dir, "report.json")
detailed_metrics = {}
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        report = json.load(f)
    detailed_metrics = report

# ── Start MLflow Run ──
with mlflow.start_run(run_name=f"profile-{result.model_name}") as run:
    
    # Log Parameters
    mlflow.log_param("model_name", result.model_name)
    mlflow.log_param("model_path", model_path)
    mlflow.log_param("platform", "simulator")
    mlflow.log_param("accelerator", "mvpv1")
    mlflow.log_param("use_simulator", True)
    
    for key in ["input_shape", "output_shape", "input_dtype", "output_dtype", 
                "accelerator_variant", "npu_toolkit_version"]:
        if key in detailed_metrics:
            mlflow.log_param(key, detailed_metrics[key])
    
    # Log Profiling Metrics
    mlflow.log_metric("arena_size_kb", result.arena_size_kb)
    mlflow.log_metric("total_macs", result.total_macs)
    
    metric_keys = {
        "flash_bytes": "flash_bytes",
        "sram_bytes": "sram_bytes",
        "op_count": "op_count",
        "layer_count": "layer_count",
        "unsupported_layer_count": "unsupported_layer_count",
        "accelerator_cycles": "accelerator_cycles",
        "mac_per_cycle": "mac_per_cycle",
    }
    for mlflow_key, report_key in metric_keys.items():
        if report_key in detailed_metrics:
            try:
                mlflow.log_metric(mlflow_key, float(detailed_metrics[report_key]))
            except (ValueError, TypeError):
                pass
    
    # Log Artifacts (stored in the volume path)
    if os.path.exists(model_path):
        mlflow.log_artifact(model_path, artifact_path="model")
    
    if os.path.isdir(result.output_dir):
        mlflow.log_artifacts(result.output_dir, artifact_path="profiling_output")
    
    # Tags for easy filtering
    mlflow.set_tag("task", "model_profiling")
    mlflow.set_tag("model_type", "tflite")
    mlflow.set_tag("accelerator", "mvpv1")
    
    print(f"✓ MLflow Run ID:     {run.info.run_id}")
    print(f"✓ Experiment:        {experiment_name}")
    print(f"✓ Artifact Location: {volume_path}")
    print(f"✓ Metrics logged:    arena_size_kb={result.arena_size_kb}, total_macs={result.total_macs:,}")
    print(f"✓ Artifacts logged:  model .tflite + profiling outputs")
    print(f"\n  View run: {mlflow.get_tracking_uri()}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

### Upload To Model Registry
Once your seleted model is successfully profiled and validated, run the below scripts (cells 10-12) to upload the model to the model registry.

In [0]:
def register_single_file(file_path, uc_model_name, run_name):
    import mlflow, os
    from mlflow import MlflowClient
    import numpy as np
    from mlflow.models import infer_signature
    from mlflow.pyfunc import PythonModel

    print(f"[UC] Registering: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # Ensure no active run
    if mlflow.active_run():
        mlflow.end_run()

    # ---- Minimal pyfunc so UC accepts model directory ----
    class Identity(PythonModel):
        def predict(self, context, model_input):
            return model_input

    # Dummy signature (UC requires model signature)
    dummy_input  = np.zeros((1,1), dtype="float32")
    dummy_output = np.zeros((1,1), dtype="float32")
    signature    = infer_signature(dummy_input, dummy_output)

    # ---- START RUN ----
    with mlflow.start_run(run_name=run_name) as run:
        # Log as a pyfunc model with the file as an extra artifact
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=Identity(),
            signature=signature,
            input_example=dummy_input,
            artifacts={"model_file": file_path},
        )

        # Register the model
        mv = mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=uc_model_name,
        )

    print(f"[UC] Registered {uc_model_name} \u2192 version {mv.version}")
    return mv.version

In [0]:
teacher_v = register_single_file(
    "/Workspace/Users/raansari@silabs.com/models/Custom_model_teacher.h5",
    "mlops_dev.default.Custom_model_teacher",
    "teacher_register"
)

In [0]:
student_v = register_single_file(
    "/Workspace/Users/raansari@silabs.com/models/Custom_model.tflite",
    "mlops_dev.default.Custom_model_student",
    "student_register"
)

#### View Logs
All the actions performed on the library will be stored locally in json format. To view them in a structured format import the logger class and view each log event type (eg. Profiling, deployment, Data Ingestion)

In [0]:
from sml.ops.logs import Logger
logger = Logger()
#logger.view()
logger.view(event_type='profiling')

#### Upload Logs
The logs files that are in json format can be uploaded and stored on to the databricks delta tables by specifying the table name and the warehouse_name. 

In [0]:
from sml.ops.logs import Logger
logger = Logger(
    warehouse_name="Serverless Starter Warehouse", 
    table_name="mlops_dev.default.logs" 
)
logger.sync_to_databricks()
